# Pokemon Type1 Predictor## ML Zoomcamp Project — Predict Pokemon Type from Name

In [ ]:
import warningswarnings.filterwarnings("ignore")import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.model_selection import train_test_splitfrom sklearn.preprocessing import LabelEncoderimport replt.style.use("seaborn-v0_8-whitegrid")plt.rcParams["figure.figsize"] = (12, 6)plt.rcParams["font.size"] = 12print("Libraries loaded!")

---

### 1. Charger les donnees

In [ ]:
df = pd.read_csv("data/pokemon.csv")print(f"Dataset brut: {df.shape[0]} lignes, {df.shape[1]} colonnes")print(f"Colonnes: {list(df.columns)}")print(df.head(10).to_string())

### Valeurs manquantes

In [ ]:
print(df.isnull().sum())type2_null_pct = df["Type2"].isnull().mean() * 100print(f"Pourcentage de Type2 vide: {type2_null_pct:.1f}%")

---

### 2. Analyse de la Variable Cible (`Type1`)

In [ ]:
df["Type1"] = df["Type1"].str.strip().str.lower().str.capitalize()df = df.dropna(subset=["Type1"])df = df[df["Type1"] != ""]types = sorted(df["Type1"].unique())print(f"Types uniques ({len(types)}): {types}")type_counts = df["Type1"].value_counts()print("Distribution des types:")print(type_counts.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))palette = [f"C{i}" for i in range(len(types))]bars = axes[0].barh(type_counts.index[::-1], type_counts.values[::-1], color=palette)axes[0].set_ylabel("Type1")axes[0].set_xlabel("Nombre de Pokemon")axes[0].set_title("Distribution du Type1", fontweight="bold")for bar, val in zip(bars, type_counts.values[::-1]):    axes[0].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2, str(val), va="center")top_n = min(10, len(type_counts))top_types = type_counts.head(top_n)other_count = type_counts[top_n:].sum()if other_count > 0:    pie_data = pd.concat([pd.Series(top_types.values, index=top_types.index),                           pd.Series([other_count], index=["Autres"])])else:    pie_data = top_typesaxes[1].pie(pie_data.values, labels=pie_data.index, autopct="%1.1f%%",            colors=plt.cm.Set3(np.linspace(0, 1, len(pie_data))), startangle=90)axes[1].set_title("Top 10 Types (+ Autres)", fontweight="bold")plt.tight_layout()plt.savefig("type_distribution.png", dpi=150, bbox_inches="tight")plt.show()

---

### 3. Feature Engineering depuis les noms

In [ ]:
df_clean = df[["Name", "Type1"]].copy()df_clean["Name"] = df_clean["Name"].astype(str).str.lower().str.strip()vowels = set("aeiou")df_clean["name_length"] = df_clean["Name"].apply(len)df_clean["vowel_ratio"] = df_clean["Name"].apply(lambda x: sum(1 for c in x if c in vowels) / max(len(x), 1))df_clean["consonant_ratio"] = df_clean["Name"].apply(lambda x: sum(1 for c in x if c.isalpha() and c not in vowels) / max(len(x), 1))df_clean["ends_with_vowel"] = df_clean["Name"].apply(lambda x: int(x[-1] in vowels))df_clean["has_double_letter"] = df_clean["Name"].apply(lambda x: int(bool(re.search(r"([a-z])\1", x))))df_clean["unique_letter_ratio"] = df_clean["Name"].apply(lambda x: len(set(x)) / max(len(x), 1))df_clean["first_letter"] = df_clean["Name"].str[0]df_clean["contains_digit"] = df_clean["Name"].apply(lambda x: int(any(c.isdigit() for c in x)))def est_syl(name):    vc = sum(1 for i in range(len(name)) if name[i] in vowels and (i == 0 or name[i-1] not in vowels) and (i == len(name)-1 or name[i+1] not in vowels))    return max(1, vc)df_clean["syllable_estimate"] = df_clean["Name"].apply(est_syl)df_clean["name_length_squared"] = df_clean["name_length"] ** 2feature_cols = ["name_length", "vowel_ratio", "consonant_ratio", "ends_with_vowel",               "has_double_letter", "unique_letter_ratio", "contains_digit",               "syllable_estimate", "name_length_squared"]print("Features extraites!")print(df_clean[["Name", "Type1", "name_length", "vowel_ratio", "has_double_letter"]].head(15).to_string())

### Box plot: Longueur du nom par type

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))sns.boxplot(data=df_clean, x="Type1", y="name_length", ax=axes[0], palette="Set2")axes[0].tick_params(axis="x", rotation=45)axes[0].set_xlabel("Type1")axes[0].set_ylabel("Longueur du Nom")axes[0].set_title("Box Plot: Longueur du Nom par Type", fontweight="bold")sns.violinplot(data=df_clean.head(500), x="Type1", y="name_length", ax=axes[1], palette="Set2")axes[1].tick_params(axis="x", rotation=45)axes[1].set_title("Violon: Distribution Longueur par Type", fontweight="bold")plt.tight_layout()plt.savefig("name_length_by_type.png", dpi=150, bbox_inches="tight")plt.show()

### Features analysis per type

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))grouped = df_clean.groupby("Type1")[["vowel_ratio", "consonant_ratio"]].mean().reset_index().melt(id_var="Type1", value_vars=["vowel_ratio", "consonant_ratio"])sns.barplot(data=grouped, x="Type1", y="value", hue="variable", ax=axes[0], palette=["#ff6b6b", "#4ecdc4"])axes[0].tick_params(axis="x", rotation=45)axes[0].set_title("Ratio Voyelles/Consonnes par Type", fontweight="bold")axes[0].set_ylabel("Ratio")dbl_pct = df_clean.groupby("Type1")["has_double_letter"].mean() * 100dbl_pct = dbl_pct.sort_values(ascending=True)sns.barplot(x=dbl_pct.values, y=dbl_pct.index, palette="Blues_d", ax=axes[1])axes[1].set_title("Pokemon avec Lettres Doublées (%) par Type", fontweight="bold")axes[1].set_xlabel("Pourcentage (%)")plt.tight_layout()plt.savefig("features_analysis.png", dpi=150, bbox_inches="tight")plt.show()

---

### 4. Feature Importance (Random Forest)

In [ ]:
from sklearn.ensemble import RandomForestClassifierfrom sklearn.preprocessing import OneHotEncoderX = df_clean[feature_cols].copy()y = df_clean["Type1"]encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)cat_encoded = encoder.fit_transform(X[["first_letter"]])cat_df = pd.DataFrame(cat_encoded, columns=encoder.get_feature_names_out(["first_letter"]))X_final = pd.concat([X.drop("first_letter", axis=1), cat_df], axis=1)rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)rf.fit(X_final, y)importance = pd.Series(rf.feature_importances_, index=X_final.columns).sort_values(ascending=True)importance.plot(kind="barh", color="steelblue")plt.xlabel("Importance")plt.ylabel("Feature")plt.title("Feature Importance (Random Forest)", fontweight="bold")plt.tight_layout()plt.savefig("feature_importance.png", dpi=150, bbox_inches="tight")plt.show()print("Top 5 features:")print(importance.tail(5).to_string())


---

### Conclusion

In [ ]:
print("Samples:", len(df))print("Classes:", len(types))print("Meilleur modele:", best_name)print("Test Accuracy:", round(results[best_name]["accuracy"], 4))print("Test F1-macro:", round(results[best_name]["f1_macro"], 4))print()print("Prochaines etapes:")print("1. python train.py -> sauvegarde model.pkl")print("2. python predict.py -> API Flask")print("3. docker build/run -> containerisation")